# LBM fluid iteration notebook

This notebook ports the JS fallback fluid solver from `playground/fluid_playground.js` into Python so you can iterate on the fluid math without the brush or browser UI. The Python helper mirrors the fallback model's `step_lbm`, `build_velocity_field`, `advance_particle`, and mask behavior.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "notebooks" / "lbm_fluid_model.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "notebooks"))

import matplotlib.pyplot as plt

from lbm_fluid_model import FluidParams, NotebookLBMFluidModel


In [ ]:
DEFAULT_WIDTH = 180
DEFAULT_HEIGHT = 120


def make_model(model_cls=NotebookLBMFluidModel, seed=7):
    params = FluidParams(
        particle_radius=4.0,
        viscosity=0.45,
        density=0.70,
        surface_tension=0.58,
        time_step=1.0,
        substeps=3,
        motion_decay=0.12,
        stop_speed=0.03,
        resolution_scale=1.0,
        fluid_scale=1.0,
        simulation_type="lbm",
    )
    model = model_cls(DEFAULT_WIDTH, DEFAULT_HEIGHT, params, random_seed=seed)
    model.seed_disc(90, 60, count=420, radius=20, initial_speed=0.35)
    return model


def run_model(model, steps=72, dt=1 / 60):
    for _ in range(steps):
        model.step(dt)
    return model


def plot_state(model, title, quiver_stride=10):
    density = model.build_density_grid(normalize=True)
    speed = model.build_speed_grid(normalize=True)
    particles = model.particles()
    xs, ys, us, vs = model.quiver(stride=quiver_stride)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    axes[0].imshow(density, origin="lower", cmap="magma")
    axes[0].set_title(f"{title} density")

    axes[1].imshow(speed, origin="lower", cmap="viridis")
    axes[1].set_title(f"{title} speed")

    axes[2].scatter([p.x for p in particles], [p.y for p in particles], s=6, alpha=0.45)
    axes[2].quiver(xs, ys, us, vs, alpha=0.55, scale=30)
    axes[2].set_xlim(0, model.display_width)
    axes[2].set_ylim(0, model.display_height)
    axes[2].set_title(f"{title} particles + flow")

    for axis in axes:
        axis.set_xticks([])
        axis.set_yticks([])
    plt.show()


In [ ]:
baseline = run_model(make_model())
plot_state(baseline, "Baseline JS-port LBM")


## Edit the fluid functions here

Override `build_velocity_field()`, `step_lbm()`, or `advance_particle()` and then rerun the comparison cell below.


In [ ]:
class ExperimentalFluidModel(NotebookLBMFluidModel):
    def step_lbm(self, step_dt):
        # Direct Python port of the JS fallback `_stepLbm`.
        # Edit this method to iterate on the fluid behavior.
        self.build_velocity_field(flow_scale=0.34, swirl_scale=0.30)
        relaxation = 0.18 + self.params.viscosity * 0.34
        pressure = 0.2 + self.params.density * 0.5
        decay = max(0.0, min(1.0, 1 - self.params.motion_decay * step_dt * 36))
        for particle in self.particles_internal:
            field_x, field_y = self.sample_velocity_field_internal(particle.x, particle.y)
            swirl_x = -field_y * pressure
            swirl_y = field_x * pressure
            particle.vx += (field_x + swirl_x - particle.vx) * relaxation
            particle.vy += (field_y + swirl_y - particle.vy) * relaxation
            particle.vx *= decay
            particle.vy *= decay
            self.advance_particle(particle, step_dt)


In [ ]:
experimental = run_model(make_model(ExperimentalFluidModel))
plot_state(experimental, "Experimental LBM")
